 Mount Drive & Import Config

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/sentiment-robustness-id/src')
from config import *

print(f'ROOT: {ROOT}')
print('✅ Config loaded')

Mounted at /content/drive
ROOT: /content/drive/MyDrive/sentiment-robustness-id
✅ Config loaded


Import Library

In [2]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

print('✅ Library siap')

✅ Library siap


Load Dataset

In [3]:
df = pd.read_csv(RAW_DATA_PATH, sep='\t')

print(f'Shape awal : {df.shape}')
print(f'Columns    : {df.columns.tolist()}')
df.head(3)

Shape awal : (23644, 4)
Columns    : ['Date', 'User', 'Tweet', 'sentiment']


,Date,User,Tweet,sentiment
0,2022-03-31 14:32:04+00:00,pikobar_jabar,Ketahui informasi pembagian #PPKM di wilayah J...,1
1,2022-03-31 09:26:00+00:00,inewsdotid,Tempat Ibadah di Wilayah PPKM Level 1 Boleh Be...,1
2,2022-03-31 05:02:34+00:00,vdvc_talk,"Juru bicara Satgas Covid-19, Wiku Adisasmito m...",1


Hapus Duplikat & Null

In [4]:
before = len(df)

df = df.drop_duplicates(subset=[TEXT_COLUMN])
df = df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])
df = df.reset_index(drop=True)

after = len(df)
print(f'Sebelum : {before:,} baris')
print(f'Sesudah : {after:,} baris')
print(f'Dihapus : {before - after:,} baris (duplikat/null)')

Sebelum : 23,644 baris
Sesudah : 23,486 baris
Dihapus : 158 baris (duplikat/null)


Fungsi Cleaning Teks

In [5]:
def clean_text(text: str) -> str:
    """
    Cleaning pipeline untuk tweet Bahasa Indonesia.
    Urutan penting: URL dulu sebelum karakter khusus.
    """
    text = str(text)

    # 1. Hapus URL
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # 2. Hapus mention (@username)
    text = re.sub(r'@\w+', '', text)

    # 3. Hapus hashtag simbol, pertahankan kata
    #    Contoh: #PPKM → PPKM
    text = re.sub(r'#(\w+)', r'\1', text)

    # 4. Lowercase
    text = text.lower()

    # 5. Hapus karakter non-alfanumerik kecuali spasi
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # 6. Hapus angka berdiri sendiri
    text = re.sub(r'\b\d+\b', '', text)

    # 7. Normalisasi spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    return text

Terapkan Cleaning & Validasi

In [6]:
df['tweet_clean'] = df[TEXT_COLUMN].apply(clean_text)

# Hapus baris yang setelah cleaning menjadi kosong atau terlalu pendek
df = df[df['tweet_clean'].str.split().str.len() >= 2]
df = df.reset_index(drop=True)

print(f'Total setelah cleaning: {len(df):,} baris')
print('\nContoh hasil cleaning:')
sample = df[[TEXT_COLUMN, 'tweet_clean']].head(5)
for _, row in sample.iterrows():
    print(f'  BEFORE: {row[TEXT_COLUMN][:80]}')
    print(f'  AFTER : {row["tweet_clean"][:80]}')
    print()

Total setelah cleaning: 23,300 baris

Contoh hasil cleaning:
  BEFORE: Ketahui informasi pembagian #PPKM di wilayah Jabar berdasarkan level 3, 2 dan 1 
  AFTER : ketahui informasi pembagian ppkm di wilayah jabar berdasarkan level dan di pikod

  BEFORE: Tempat Ibadah di Wilayah PPKM Level 1 Boleh Berkapasitas 100 Persen. Baca Seleng
  AFTER : tempat ibadah di wilayah ppkm level boleh berkapasitas persen baca selengkapnya 

  BEFORE: Juru bicara Satgas Covid-19, Wiku Adisasmito menjelaskan bahwa bukber diperboleh
  AFTER : juru bicara satgas covid wiku adisasmito menjelaskan bahwa bukber diperbolehkan 

  BEFORE: Ketahui informasi pembagian #PPKM di wilayah Jabar berdasarkan level 4, 3, dan 2
  AFTER : ketahui informasi pembagian ppkm di wilayah jabar berdasarkan level dan di pikod

  BEFORE: Kementerian Agama menerbitkan Surat Edaran Nomor 06/2022 tentang Pelaksanaan Keg
  AFTER : kementerian agama menerbitkan surat edaran nomor tentang pelaksanaan kegiatan pe



Stratified Split 70 / 15 / 15

In [7]:
from sklearn.model_selection import train_test_split

X = df['tweet_clean'].values
y = df[LABEL_COLUMN].values

# Split 1: pisahkan test set (15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

# Split 2: pisahkan val dari sisa (15% dari total ≈ 17.6% dari temp)
val_ratio = VAL_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=val_ratio,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print('=== SPLIT RESULT ===')
print(f'Train : {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)')
print(f'Val   : {len(X_val):,} ({len(X_val)/len(X)*100:.1f}%)')
print(f'Test  : {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)')

print('\n=== DISTRIBUSI LABEL PER SPLIT ===')
for split_name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    counts = pd.Series(y_split).value_counts().sort_index()
    dist   = ' | '.join([f'{LABEL_MAP[i]}: {counts[i]} ({counts[i]/len(y_split)*100:.1f}%)' for i in sorted(counts.index)])
    print(f'  {split_name:6s}: {dist}')

=== SPLIT RESULT ===
Train : 16,310 (70.0%)
Val   : 3,495 (15.0%)
Test  : 3,495 (15.0%)

=== DISTRIBUSI LABEL PER SPLIT ===
  Train : Negatif: 1361 (8.3%) | Positif: 12192 (74.8%) | Netral: 2757 (16.9%)
  Val   : Negatif: 291 (8.3%) | Positif: 2613 (74.8%) | Netral: 591 (16.9%)
  Test  : Negatif: 291 (8.3%) | Positif: 2613 (74.8%) | Netral: 591 (16.9%)


Simpan Processed Data

In [8]:
# Gabungkan semua split ke satu file dengan kolom 'split'
df_train = pd.DataFrame({'tweet_clean': X_train, LABEL_COLUMN: y_train, 'split': 'train'})
df_val   = pd.DataFrame({'tweet_clean': X_val,   LABEL_COLUMN: y_val,   'split': 'val'})
df_test  = pd.DataFrame({'tweet_clean': X_test,  LABEL_COLUMN: y_test,  'split': 'test'})

df_clean = pd.concat([df_train, df_val, df_test], ignore_index=True)
df_clean.to_csv(PROCESSED_DATA_PATH, index=False)

print(f'✅ Saved: {PROCESSED_DATA_PATH}')
print(f'   Shape : {df_clean.shape}')
print(f'   Kolom : {df_clean.columns.tolist()}')
print(f'\nPreview:')
df_clean.head(3)

✅ Saved: /content/drive/MyDrive/sentiment-robustness-id/data/processed/data_clean.csv
   Shape : (23300, 3)
   Kolom : ['tweet_clean', 'sentiment', 'split']

Preview:


,tweet_clean,sentiment,split
0,indonesia jakarta jokowi vaksin ppkm economy b...,1,train
1,aturan baru di jakarta makan di warteg wajib t...,1,train
2,tegas bgt ya singapura indonesia jg kok sebetu...,2,train


Verifikasi File Tersimpan

In [9]:
import os

if os.path.exists(PROCESSED_DATA_PATH):
    df_verify = pd.read_csv(PROCESSED_DATA_PATH)
    print(f'✅ File berhasil disimpan dan dapat dibaca kembali')
    print(f'   Shape  : {df_verify.shape}')
    print(f'   Splits : {df_verify["split"].value_counts().to_dict()}')
else:
    print('❌ File tidak ditemukan, cek path!')

✅ File berhasil disimpan dan dapat dibaca kembali
   Shape  : (23300, 3)
   Splits : {'train': 16310, 'val': 3495, 'test': 3495}


Ringkasan Preprocessing

In [10]:
print('=' * 50)
print('     RINGKASAN PREPROCESSING')
print('=' * 50)

df_train_only = df_clean[df_clean['split'] == 'train']
df_val_only   = df_clean[df_clean['split'] == 'val']
df_test_only  = df_clean[df_clean['split'] == 'test']

print(f'''
📋 CLEANING
  URL, mention, hashtag  → dihapus
  Lowercase              → diterapkan
  Karakter non-alfabet   → dihapus
  Tweet terlalu pendek   → dibuang

📂 SPLIT
  Train : {len(df_train_only):,} baris
  Val   : {len(df_val_only):,} baris
  Test  : {len(df_test_only):,} baris
  Metode: Stratified Split (seed={RANDOM_STATE})

💾 OUTPUT
  {PROCESSED_DATA_PATH}

✅ Lanjut ke 03_Noise_Injection.ipynb
  → Test set akan dibuat dalam 4 versi:
     clean | noise 10% | noise 20% | noise 30%
''')
print('=' * 50)

     RINGKASAN PREPROCESSING

📋 CLEANING
  URL, mention, hashtag  → dihapus
  Lowercase              → diterapkan
  Karakter non-alfabet   → dihapus
  Tweet terlalu pendek   → dibuang

📂 SPLIT
  Train : 16,310 baris
  Val   : 3,495 baris
  Test  : 3,495 baris
  Metode: Stratified Split (seed=42)

💾 OUTPUT
  /content/drive/MyDrive/sentiment-robustness-id/data/processed/data_clean.csv

✅ Lanjut ke 03_Noise_Injection.ipynb
  → Test set akan dibuat dalam 4 versi:
     clean | noise 10% | noise 20% | noise 30%

